In [1]:
%cd ../

/nas/zhangtianning.di/projects/unique_data_build


In [2]:
html_path = "/nvme/zhangtianning.di/sharefold/whole_arxiv_all_files/archive_html/2009/2009.11140/2009.11140.html"

In [3]:
from uparxive.html_to_json_nougat.nougat.dataset.parser.latexml_parser import *

In [4]:
from uparxive.html_to_json_nougat.nougat.dataset.parser.html2md import *

In [5]:
from bs4 import BeautifulSoup
html = BeautifulSoup(
    htmlmin.minify(
        open(html_path, "r", encoding="utf-8").read().replace("\xa0", " "),
        remove_all_empty_space=1,
    ),
    features="html.parser",
)


In [6]:
from uparxive.html_to_json_nougat.html_to_dense_text import *

In [18]:
from uparxive.html_to_json_nougat.html_to_markdown import *

In [25]:
args=HTMLtoMDConfig(root_path="")

In [29]:
tmp_html_path = html_path
output_dir='test'

In [35]:
args.passNote=True

In [ ]:
#############
use_count_type_ref  = not args.use_origin_ref_number
reterive_result_mode=args.reterive_result_mode
_paper_id =os.path.basename(tmp_html_path.replace('.html',''))
paper_id = f"ArXiv.{_paper_id}"
with open(tmp_html_path,'r',encoding='utf-8', errors='ignore') as f:
    soup_whole = BeautifulSoup(
        htmlmin.minify(
            f.read().replace("\xa0", " "),
            remove_all_empty_space=1,
        ),
        features="html.parser",
        )


soup = soup_whole.article
ReferenceDir= os.path.join(output_dir, "Reference")


ref_count = retrieve_all_cite(soup)
new_soup = deepcopy(soup)
new_soup,reference_labels,bibitem_ref_metadata,note_ref_labels, note_ref_metadata= remove_entire_bibliography_and_build_labels(new_soup,ref_count, args)
if len(note_ref_metadata)>5:
    for key,val in note_ref_metadata.items():
        logging.info(f"{key} ==> [ {better_latex_sentense_string(' '.join(val[1].text))} ]")
    checkTooManyNote(f'the note_ref_metadata num={len(note_ref_metadata) } is too much , please check the file {tmp_html_path}')
    logging.warning('WARNING:Too Many note, we roll back to no note mode')
    args.passNote = True
    soup,reference_labels,bibitem_ref_metadata,note_ref_labels, note_ref_metadata= remove_entire_bibliography_and_build_labels(soup,ref_count,args)
else:
    soup= new_soup

reference_labels, reference_labels_not_in_context = divide_the_dict_into_two_part_by_keys(reference_labels, ref_count)
note_ref_labels, note_ref_labels_not_in_context = divide_the_dict_into_two_part_by_keys(note_ref_labels,ref_count)
bibitem_ref_metadata, bibitem_ref_metadata_not_in_context = divide_the_dict_into_two_part_by_keys(bibitem_ref_metadata,ref_count)
note_ref_metadata, note_ref_metadata_not_in_context = divide_the_dict_into_two_part_by_keys(note_ref_metadata,ref_count)

footnote_labels, footnote_metadata = remove_note_and_record_the_infomration(soup)
note_ref_labels=note_ref_labels|footnote_labels
note_ref_metadata=note_ref_metadata|footnote_metadata

put_back_keys = put_note_string_back_into_each_sentence(soup,ref_count,note_ref_metadata)
## since we put those key back into content, we never need those key anymore and wont save them in the reference.txt
for key in put_back_keys:
    del note_ref_labels[key]
    del note_ref_metadata[key]
# notice, after this line, the key in note_ref_metadata and note_ref_labels is different

figures_labels, figures_metadata = remove_figures_record_the_labels(soup)
tables_labels, tables_metadata   = remove_tables_record_the_labels(soup)
floats_labels, floats_metadata   = remove_floats_record_the_labels(soup)
equation_group_labels = {}
equation_labels = {}

in_content_ref_labels = {
    'Figure':figures_labels,
    'Table':tables_labels,
    'Equation':equation_labels,
    'Equationgroup':equation_group_labels,
    'Floats':floats_labels
}
labels               = collect_tags_and_record_all_labels(soup)## like section and so one


all_citation_keys = set(ref_count)
all_reference_keys= (set(reference_labels)|
                     set(note_ref_labels)|
                     set(figures_labels)|
                     set(tables_labels)|
                     set(equation_labels)|
                     set(equation_group_labels)|
                     set(equation_group_labels)|
                     set(floats_labels))

for val_pool in labels.values():
    all_reference_keys = all_reference_keys | set(val_pool)
missing_citation = all_citation_keys - all_reference_keys
missing_citation_labels = {missing_citation_label:f'MissingCite_{i}' for i,missing_citation_label in enumerate(missing_citation)}


#assert len(bibitem_ref_metadata)>0, f"Error: this file [{tmp_xml_path}] donts have bib???"

if reterive_result_mode:
    assert os.path.exists(os.path.join(ReferenceDir,'reference.keys.done'))
    assert os.path.getsize(os.path.join(ReferenceDir,'reference.txt')) == 0, "if you want to inject the reterive result, please make sure all the element is reterived"
    with open(os.path.join(ReferenceDir,'reference.keys.done'),'r') as f:
        reference_keys = [t.strip() for t in f]
    with open(os.path.join(ReferenceDir,'reference.es_retrived_citation.json.done'),'r') as f:
        reference_reterives = json.load(f)
    assert len(reference_keys) == len(reference_reterives), "the reterive result should have the same length as the keys"
    new_label_mapping = {}
    for key, reterive_result in zip(reference_keys,reference_reterives):
        if key not in new_label_mapping:new_label_mapping[key] = []
        new_label_mapping[key].append(get_unique_id_from_reterive_result(reterive_result))
    for key in new_label_mapping.keys():
        new_label_mapping[key] = "<"+ ",".join(new_label_mapping[key]) + ">"
    reference_labels = new_label_mapping


whole_ref_to_labels = collect_whole_reference(in_content_ref_labels|
                                              {'Reference':reference_labels,'Missing':missing_citation_labels}|
                                              labels, 
                                              use_count_type_ref=use_count_type_ref)

lack_ref = list(set(ref_count) - (set(all_reference_keys)|set(whole_ref_to_labels)))
if len(lack_ref)>0:
    logging.info(f'you have {len(lack_ref)} ref lacks, such as {lack_ref[:4]}, please check the file {tmp_html_path}')
    raise MisMatchRefError

## now, the left note metadata is those string looks like a citation, and we will put them back into the bibitem information
for remain_key, remain_val in note_ref_metadata.items():

    reference_labels[remain_key]=note_ref_labels[remain_key]
    string = cleanup_reference_string(remain_val[1], whole_ref_to_labels,paper_id, refs_that_wont_recovery=put_back_keys)
    bibitem_ref_metadata[remain_key]=better_latex_sentense_string(string)


whole_ref_to_labels = collect_whole_reference(in_content_ref_labels|
                                              {'Reference':reference_labels,'Missing':missing_citation_labels}|
                                              labels, 
                                              use_count_type_ref=use_count_type_ref)

recovery_whole_citation_complete(soup,whole_ref_to_labels, paper_id,refs_that_wont_recovery=[])


for remain_key, remain_val in note_ref_metadata_not_in_context.items():
    string = cleanup_reference_string(remain_val[1], whole_ref_to_labels,paper_id, refs_that_wont_recovery=put_back_keys)
    note_ref_metadata_not_in_context[remain_key]=better_latex_sentense_string(string)
    ## do this again since we modify the bibitem_ref_metadata

for metadatapool in [figures_metadata, tables_metadata, floats_metadata]:
    for remain_key, remain_val in metadatapool.items():
        string = cleanup_reference_string(remain_val, whole_ref_to_labels,paper_id, refs_that_wont_recovery=put_back_keys)
        metadatapool[remain_key]=better_latex_sentense_string(string)


whole_metadata = {'figures_metadata':figures_metadata,
                  'tables_metadata':tables_metadata,
                  'floats_metadata':floats_metadata,
                  'bibitem_ref_metadata':bibitem_ref_metadata,}

In [42]:
content_soup = copy.deepcopy(soup)
appendix_content = []
if content_soup.find(class_='ltx_appendix'):
    appendix_soup = content_soup.find(class_='ltx_appendix')
    appendix_content = collect_sections_to_content(appendix_soup, args)
    #appendix_soup.decompose()
elif content_soup.find(class_='ltx_part'):
    appendix_soup = content_soup.find(class_='ltx_part')
    appendix_content = collect_sections_to_content(appendix_soup, args)
    #appendix_soup.decompose()

In [45]:
appendix_content

['Mathieu Florence\n\nAbstract. Let $p$ be a prime. In this paper, we investigate the existence of liftings of mod $p$ representations of a profinite group, to mod $p^{2}$ representations. As a concrete application of general results, we get the following. Let $F$ be a field, with separable closure $F_{s}$, and let $d\\geq 1$ an integer. Then, every Galois representation\n\n$$\n\\rho_{1}:\\mathrm{Gal}(F_{s}/F)\\longrightarrow\\mathbf{GL}_{d}(\\mathbb{Z}/p)\n$$\n\nlifts to\n\n$$\n\\rho_{2}:\\mathrm{Gal}(F_{s}/F)\\longrightarrow\\mathbf{GL}_{d}(\\mathbb{Z}/p^{2}).\n$$\n\nThis is a vast improvement on previously known results- see the Introduction for details. To achieve it, we work in the framework of cyclotomic pairs and of smooth profinite groups, developped in[Ref.[0] of ArXiv.2009.11140], and prove a much deeper result. Namely, complete flags of mod$p$ semi-linear representations of a $(1,1)$-smooth profinite group lift, step by step, modulo $p^{2}$. This is Theorem (See [Theorem.53 

In [32]:
args.outputmd = True

In [38]:
sections_contnet = collect_sections_to_content(content_soup, args)

In [40]:
content_soup = copy.deepcopy(soup)

In [37]:
print(collect_sections_to_content(soup, args))

['##\n\nMathieu Florence\n\nAbstract. Let $p$ be a prime. In this paper, we investigate the existence of liftings of mod $p$ representations of a profinite group, to mod $p^{2}$ representations. As a concrete application of general results, we get the following. Let $F$ be a field, with separable closure $F_{s}$, and let $d\\geq 1$ an integer. Then, every Galois representation\n\n$$\n\\rho_{1}:\\mathrm{Gal}(F_{s}/F)\\longrightarrow\\mathbf{GL}_{d}(\\mathbb{Z}/p)\n$$\n\nlifts to\n\n$$\n\\rho_{2}:\\mathrm{Gal}(F_{s}/F)\\longrightarrow\\mathbf{GL}_{d}(\\mathbb{Z}/p^{2}).\n$$\n\nThis is a vast improvement on previously known results- see the Introduction for details. To achieve it, we work in the framework of cyclotomic pairs and of smooth profinite groups, developped in[Ref.[0] of ArXiv.2009.11140], and prove a much deeper result. Namely, complete flags of mod$p$ semi-linear representations of a $(1,1)$-smooth profinite group lift, step by step, modulo $p^{2}$. This is Theorem (See [Theor